In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import filtfilt
import matplotlib.gridspec as gridspec
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 加载分组结果和滤波器参数
groups_df = pd.read_csv(r'../数据/噪声分组.csv')
filter_params_df = pd.read_csv(r'../数据/滤波器参数设置.csv')

print("\n[1] 加载分组结果:")
print(groups_df[['code', 'group', 'noise_score']].head())

print("\n[2] 加载滤波器参数:")
print(filter_params_df[['code', 'group', 'cutoff_period', 'filter_order', 'filter_type']].head())

# ===================== 读取 300 只股票在同一个文件 =====================
stock_file = r"../数据/沪深300成分股_10年.csv"
df_all = pd.read_csv(stock_file)

# 日期格式 + 排序
df_all['trade_date'] = pd.to_datetime(df_all['trade_date'])
df_all = df_all.sort_values(['ts_code', 'trade_date']).reset_index(drop=True)

# 按股票代码分组，存入字典
stock_data = {}
for code, group in df_all.groupby('ts_code'):
    df = group.copy().reset_index(drop=True)
    
    # 清洗收盘价
    df['close'] = pd.to_numeric(df['close'], errors='coerce')
    df = df.dropna(subset=['close'])
    
    # 只保留需要的股票（在 groups_df 里的）
    if code in groups_df['code'].tolist():
        stock_data[code] = df
        print(f"加载 {code}: {len(df)} 条记录")

print(f"\n✅ 成功加载 {len(stock_data)} 只股票数据")


[1] 加载分组结果:
        code group  noise_score
0  600522.SH  高噪声组     0.722572
1  688256.SH  高噪声组     0.707643
2  603993.SH  高噪声组     0.695328
3  688126.SH  高噪声组     0.688988
4  300316.SZ  高噪声组     0.669862

[2] 加载滤波器参数:
        code group  cutoff_period  filter_order filter_type
0  600522.SH  高噪声组           60.0             6   chebyshev
1  688256.SH  高噪声组           60.0             6   chebyshev
2  603993.SH  高噪声组           60.0             6   chebyshev
3  688126.SH  高噪声组           60.0             6   chebyshev
4  300316.SZ  高噪声组           60.0             6   chebyshev
加载 000001.SZ: 2427 条记录
加载 000002.SZ: 2336 条记录
加载 000063.SZ: 2356 条记录
加载 000100.SZ: 2259 条记录
加载 000157.SZ: 2426 条记录
加载 000166.SZ: 2401 条记录
加载 000301.SZ: 2253 条记录
加载 000333.SZ: 2377 条记录
加载 000338.SZ: 2425 条记录
加载 000408.SZ: 2386 条记录
加载 000425.SZ: 2392 条记录
加载 000538.SZ: 2263 条记录
加载 000568.SZ: 2409 条记录
加载 000596.SZ: 2427 条记录
加载 000617.SZ: 2316 条记录
加载 000625.SZ: 2407 条记录
加载 000630.SZ: 2408 条记录
加载 000651.SZ: 2276 条记录
加载 0006

In [5]:
#设计低通滤波器
import scipy.signal as signal#滤波器算法的导入
def design_lowpass_filter(cutoff_freq, fs=1.0, order=4, filter_type='butterworth'):
    """
    参数:
    - cutoff_freq: 截止频率 (Hz)，高于这个频率将被删除
    - fs: 采样频率 (默认1，表示1天1个样本)
    - order: 滤波器阶数，阶数越高，滤波效果越陡
    - filter_type: 'butterworth' 或 'chebyshev'滤波器类型
    
    返回:
    - b, a: 滤波器系数
    """
    nyquist = fs / 2#能处理的最高有效频率 = 采样频率的一半
    normalized_cutoff = cutoff_freq / nyquist
     # 安全限制：归一化频率不能 ≥1
    if normalized_cutoff >= 1.0:
        normalized_cutoff = 0.99
    
    if filter_type == 'butterworth':
        b, a = signal.butter(order, normalized_cutoff, btype='low')
    elif filter_type == 'chebyshev':
        b, a = signal.cheby1(order, 0.5, normalized_cutoff, btype='low')
    else:
        b, a = signal.butter(order, normalized_cutoff, btype='low')
    
    return b, a

In [6]:
#应用零相位滤波器（避免相位延迟），在通过低通滤波器筛去高频的噪声信号后消去相位延迟，与原始数据对齐
def apply_filter(price_series, b, a):
    # 保留索引，避免错位
    valid_idx = price_series.dropna().index
    price_clean = price_series.loc[valid_idx].values#提取有效数据，纯数字的数组
    filtered = filtfilt(b, a, price_clean)
    
    # 返回带索引的序列
    filtered_series = price_series.copy()#保留日期和结构
    filtered_series.loc[valid_idx] = filtered#填回数据，保证对齐
    return filtered_series

In [7]:
def apply_filter_to_stock(price_series, cutoff_period, order=4, filter_type='butterworth'):
    """
    股票专用：直接传入 周期参数，自动转频率 + 滤波
    """
    # 复制一份，避免修改原数据
    price = price_series.copy()
    # 股票核心：周期 → 频率 转换
    if cutoff_period <= 1:
        cutoff_period = 2  # 防止除0
    
    cutoff_freq = 1.0 / cutoff_period  # 周期转频率
    fs = 1.0  # 日线：1天1个点

     # 记录有效索引（非 NaN 位置）
    valid_mask = price.notna()
    price_valid = price[valid_mask]  # 只拿干净数据滤波
    
    if len(price_valid) == 0:
        # 全是空值，返回全空序列
        return pd.Series(index=price.index, dtype=price.dtype), None, None

    # 设计滤波器
    b, a = design_lowpass_filter(
        cutoff_freq=cutoff_freq,
        fs=fs,
        order=order,
        filter_type=filter_type
    )
    
    # 零相位滤波
    filtered_clean = apply_filter(price_valid, b, a)
    
    # 把滤波结果还原成带日期索引的Series（保持对齐）
    filtered_series = pd.Series(index=price.index, dtype=price.dtype)
    filtered_series.loc[valid_mask[valid_mask].index] = filtered_clean#填上滤波后的数据
    
    return filtered_series, b, a

In [8]:
def evaluate_filter_performance(original, filtered, residual):#三个输入，原始股价，过滤后的股价和残差（过滤后的噪声）
    # 去无效值
    mask = original.notna() & filtered.notna()#找到原始数据不为空且滤波数据也不为空的位置
    orig = original[mask]
    filt = filtered[mask]
    res = residual[mask]
    
    # 指标
    #降噪率
    #公式：1 - 噪声 / 原始波动
    noise_reduction = 1 - (np.var(res) / np.var(orig)) if np.var(orig) != 0 else 0
    #平滑度比例
    #滤波后波动 / 原始波动
    smoothness_ratio = np.var(np.diff(filt)) / (np.var(np.diff(orig)) + 1e-8) if len(filt) > 1 else 0
    #趋势保留度
    #1 - 趋势相关系数
    trend_correlation = np.corrcoef(orig, filt)[0, 1]
    information_loss = 1 - trend_correlation#越接近1越好

    #能量保留比例
    energy_original = np.sum(orig ** 2)            # 原始信号能量
    energy_filtered = np.sum(filt ** 2)            # 滤波后信号能量
    energy_remaining = energy_filtered / (energy_original + 1e-8)  # 能量保留比例

    return {
        'noise_reduction': round(noise_reduction, 4),
        'smoothness_ratio': round(smoothness_ratio, 4),
        'trend_correlation': round(trend_correlation, 4),
        'information_loss': round(information_loss, 4),
        'energy_remaining': round(energy_remaining, 4)
    }

In [9]:
# 批量评估所有股票，使用上面定义的函数
def batch_evaluate(stock_data_dict, filter_params_dict):
    results = []
     # 用于保存所有股票的滤波结果（合并输出）
    all_filtered_results = []
    for code, data in stock_data_dict.items():
        if code not in filter_params_dict:
            continue
            
        price = data.set_index('trade_date')['close']#把日期设为索引，只拿 close 收盘价做滤波
        params = filter_params_dict[code]#取出这只股票专属的滤波参数
        
        # 应用滤波器
        filtered, b, a = apply_filter_to_stock(
            price,
            cutoff_period=params['cutoff_period'],
            order=params['filter_order'],
            filter_type=params['filter_type']
        )
        
        # 构建单只股票的完整数据表：日期 + 原始收盘价 + 滤波后收盘价 + 残差
        stock_filter_df = pd.DataFrame({
            'trade_date': price.index,
            'code': code,
            'close_price': price.values,
            'filtered_price': filtered.values,
            'residual': price.values - filtered.values,  # 残差 = 原始 - 滤波
            'industry': data['industry'].values,    # 行业
            'market_cap': data['market_cap'].values # 市值
        })
          # 把当前股票结果加入总表
        all_filtered_results.append(stock_filter_df)

        p = price.values
        f = filtered.values
        
        # 有效掩码（纯数组，无索引），解决price和filter索引对不齐的问题
        valid_mask = ~(np.isnan(p) | np.isnan(f))
        
        # 只取有效数据
        p_valid = p[valid_mask]
        f_valid = f[valid_mask]
        residual_valid = p_valid - f_valid

        # 传给评估函数（不需要索引也能计算）
        metrics = evaluate_filter_performance(
            pd.Series(p_valid), 
            pd.Series(f_valid), 
            pd.Series(residual_valid)
        )
        
        results.append({
            'code': code,
            'group': params['group'],
            'cutoff_period': params['cutoff_period'],
            'noise_reduction': metrics['noise_reduction'],
            'smoothness_ratio': metrics['smoothness_ratio'],
            'trend_correlation': metrics['trend_correlation'],
            'information_loss': metrics['information_loss'],
            'energy_remaining': metrics['energy_remaining']
        })
    if all_filtered_results:
        all_filtered_df = pd.concat(all_filtered_results, ignore_index=True)
        all_filtered_df.to_csv(r"../数据/全股票滤波结果总表.csv", index=False, encoding='utf-8-sig')
        print("✅ 全股票滤波结果已保存到：全股票滤波结果总表.csv")
    
    return pd.DataFrame(results)

In [10]:
filter_params_dict = filter_params_df.drop_duplicates('code').set_index('code').to_dict(orient='index')
result_df = batch_evaluate(stock_data, filter_params_dict)
print("\n批量评估完成：")
print(result_df)
result_df.to_csv(r"../数据/评估效果.csv")

✅ 全股票滤波结果已保存到：全股票滤波结果总表.csv

批量评估完成：
          code group  cutoff_period  noise_reduction  smoothness_ratio  \
0    000001.SZ  低噪声组      15.000000           0.9956            0.1245   
1    000002.SZ  中噪声组      25.226366           0.9843            0.0798   
2    000063.SZ  低噪声组      27.189946           0.9754            0.0739   
3    000100.SZ  高噪声组      39.155992           0.9784            0.0403   
4    000157.SZ  低噪声组      15.020037           0.9954            0.1196   
..         ...   ...            ...              ...               ...   
295  688303.SH  中噪声组      33.454548           0.9795            0.0411   
296  688396.SH  中噪声组      32.406045           0.9567            0.0414   
297  688472.SH  高噪声组      45.954348           0.8804            0.0230   
298  688506.SH  高噪声组      39.996817           0.9793            0.0287   
299  688981.SH  高噪声组      31.160557           0.9805            0.0617   

     trend_correlation  information_loss  energy_remaining  
0            